# JSMA: Jacobian and Gradients

This notebook follows the HTB Academy **Jacobian and Gradients** subsection. We build the reusable pieces needed by the Jacobian-based Saliency Map Attack (JSMA): one-class input gradients, the full Jacobian, target/competitor extraction, and search-space masking.

JSMA is targeted: it asks which input pixels can increase a chosen target logit while decreasing the other logits. The attack loop and saliency formula come in later subsections.

## Mathematical map

For class score `F_i(x)` and pixel `x_j`, the Jacobian entry is

`J_ij = partial F_i / partial x_j`

Read this aloud as: **J sub i-j equals the partial derivative of F sub i with respect to x sub j**. It means how much class `i`'s score changes when pixel `j` changes slightly. For `m` classes and `n` input features, `J` has shape `(m, n)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from htb_ai_library.core import set_reproducibility
from htb_ai_library.data import get_mnist_loaders
from htb_ai_library.models import SimpleLeNet
from htb_ai_library.training import train_model
from htb_ai_library.utils import save_model, load_model
from htb_ai_library.visualization import use_htb_style

use_htb_style()
set_reproducibility(1337)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

The next cell loads the same LeNet-like MNIST model used by HTB. A cached checkpoint avoids retraining on every run. `eval()` disables training-only behavior such as dropout.

In [ ]:
train_loader, test_loader = get_mnist_loaders(batch_size=128)
model_path = output_dir / 'mnist_target.pth'
model = SimpleLeNet().to(device)

if model_path.exists():
    print(f'Loading existing model from {model_path}')
    model = load_model(model, model_path, device)
else:
    print('Training new model...')
    model = train_model(model, train_loader, test_loader, epochs=5, learning_rate=0.001, device=device)
    save_model(model, model_path)

model.eval()
print('Model ready for JSMA gradients')

## One class gradient

Autograd needs one scalar output. Selecting `logits[0, class_idx]` means we differentiate one image's one class score with respect to every input pixel. We use logits rather than softmax probabilities so target and competitor sensitivities remain independent.

In [ ]:
def compute_class_gradient(x, model, class_idx, wrt='logits'):
    """Return d(selected class score) / d(input) as a flat NumPy vector."""
    if x.shape[0] != 1:
        raise ValueError('compute_class_gradient expects batch size 1')

    x_grad = x.detach().clone().requires_grad_(True)
    logits = model(x_grad)

    if wrt == 'logits':
        scalar = logits[0, class_idx]
    elif wrt == 'probabilities':
        probs = F.softmax(logits, dim=1)
        scalar = probs[0, class_idx]
    else:
        raise ValueError("wrt must be 'logits' or 'probabilities'")

    scalar.backward()
    return x_grad.grad.detach().cpu().numpy().flatten().copy()

In [ ]:
# A deterministic four-feature toy model makes the gradient shape visible.
model_test = nn.Sequential(nn.Flatten(), nn.Linear(4, 2)).eval()
x_test = torch.tensor([[[[0.5, 0.3], [0.2, 0.8]]]])
grad_class0 = compute_class_gradient(x_test, model_test, 0)
print('Gradient shape:', grad_class0.shape)
print('Gradient for class 0:', grad_class0)

A positive gradient component says increasing that feature raises the selected score; a negative component says decreasing it may help. Flattening preserves channel-height-width order, so the vector index maps consistently back to the image.

In [ ]:
def compute_jacobian_matrix(x, model, num_classes=10, wrt='logits'):
    """Return a (num_classes, num_features) Jacobian for one image."""
    if x.shape[0] != 1:
        raise ValueError('compute_jacobian_matrix expects batch size 1')

    rows = [
        compute_class_gradient(x, model, class_idx, wrt)
        for class_idx in range(num_classes)
    ]
    return np.asarray(rows)

print('For MNIST:', 10 * 28 * 28, 'Jacobian values')
print('Expected shape:', (10, 784))

Each Jacobian row is one class gradient. The explicit batch-size check prevents accidentally computing a result that does not correspond to one image's per-sample sensitivities.

In [ ]:
def extract_target_gradient(jacobian, target_class):
    return jacobian[target_class].copy()


def extract_other_gradients(jacobian, target_class):
    total_grad = jacobian.sum(axis=0)
    return total_grad - jacobian[target_class]

J_toy = np.array([
    [0.2, 0.5, -0.1, 0.3],
    [-0.1, 0.2, 0.4, -0.2],
    [0.6, -0.3, 0.1, 0.5],
])
target = 2
alpha = extract_target_gradient(J_toy, target)
beta = extract_other_gradients(J_toy, target)
print('Target gradient alpha:', alpha)
print('Other-gradient sum beta:', beta)

Read `alpha` as the target class's sensitivity vector. Read `beta` as the combined sensitivity of every non-target class. In later saliency code, JSMA looks for directions where alpha is positive and beta is negative (or the reverse direction).

In [ ]:
def apply_search_mask(gradient, search_space):
    """Zero unavailable features without changing their indices."""
    return gradient * search_space

grad = np.array([0.5, -0.2, 0.8, 0.1, -0.4])
mask = np.array([True, False, True, False, True])
masked_grad = apply_search_mask(grad, mask)
print('Original gradient:', grad)
print('Search-space mask:', mask)
print('Masked gradient:  ', masked_grad)

Multiplication converts `True` to 1 and `False` to 0. We keep the vector length and therefore preserve pixel-to-index correspondence. This mask can exclude pixels that are already saturated at `clip_min`/`clip_max` or have already been consumed by the attack.

**Notebook boundary:** this subsection stops here. Saliency scoring and feature selection belong to the next HTB subsection.